# Validator Agent Notebook

This notebook demonstrates the Validator Agent which validates quantum circuits and uses LLM to fix broken code.

## Features
- **Local Validation**: Runs code using the **Amazon Braket SDK** (BraketCompiler + LocalSimulator)—no external backend required
- **LLM-Powered Fixing**: When code fails, the LLM analyzes errors and suggests fixes

## Prerequisites
1. **Amazon Braket SDK** installed (`pip install amazon-braket-sdk`)
2. AWS/LLM configured in `config/config.json` for validator model (for LLM fixing)

---

# Part 1: Setup & Local Braket SDK Check

Verify the Amazon Braket SDK is available and that local compile + simulate works.

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.braket_rag_code_assistant.config import get_config, setup_logging
from src.agents.validator import ValidatorAgent
from src.tools.compiler import BraketCompiler
from src.tools.simulator import QuantumSimulator

# Setup logging
setup_logging()
print("✅ Imports completed successfully.")

2026-03-15 02:02:27 | INFO     | src.braket_rag_code_assistant.config.logging:setup_all:137 | Logging configuration completed


✅ Imports completed successfully.


## 1.1 Check Local Braket SDK

Verify the Amazon Braket SDK is installed and that we can compile and simulate a small circuit locally.

In [2]:
# Use local Braket compiler and simulator (no external backend)
compiler = BraketCompiler()
simulator = QuantumSimulator()

# Minimal Braket code: compile and run locally
sample_code = """
from braket.circuits import Circuit
circuit = Circuit().h(0).cnot(0, 1)
"""

result = compiler.compile(sample_code, execute=True)

if result.get("success") and result.get("circuit") is not None:
    print("✅ Amazon Braket SDK is available (local compile OK)")
    sim_result = simulator.simulate(result["circuit"], repetitions=100)
    if sim_result.get("success"):
        print("✅ Local simulation OK")
        print(f"   Sample counts: {sim_result.get('histogram', {})}")
    else:
        print("⚠️ Compile OK but simulation failed:", sim_result.get("error"))
else:
    print("❌ Braket SDK compile failed")
    for e in result.get("errors", []):
        print(f"   {e}")

2026-03-15 02:02:27 | INFO     | config.config_loader:load:101 | ✅ Loaded configuration from D:\University\Uni\Hackathon\Amazons\config\config.json
2026-03-15 02:02:27 | INFO     | src.tools.qcanvas_client:__init__:33 | QCanvasClient initialized with URL: http://localhost:8000


Backend URL: http://localhost:8000
Timeout: 30s



2026-03-15 02:02:31 | WARNING  | src.tools.qcanvas_client:check_health:166 | Health check failed: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


❌ QCanvas backend is NOT AVAILABLE!

Please start the backend:
  cd ../backend
  python start.py


## 1.2 Local validation path

The Validator Agent runs code **locally** via `BraketCompiler` and `QuantumSimulator` (Amazon Braket SDK). No QCanvas or other backend is required.

In [3]:
# test_code used later in Part 2 for validation examples
test_code = """
from braket.circuits import Circuit
circuit = Circuit().h(0).cnot(0, 1)
print(circuit)
"""
print("✅ Part 1 complete. Validation will use the Amazon Braket SDK (local compiler + simulator).")

📤 Sending code to /api/converter/convert...



2026-03-15 02:02:35 | ERROR    | src.tools.qcanvas_client:convert_to_qasm:69 | Conversion error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/converter/convert (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


Success: False

❌ Conversion failed!
Error: Conversion error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/converter/convert (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


## 1.3 Part 1 summary

No external backend is needed. The Validator Agent uses **BraketCompiler** and **QuantumSimulator** (Amazon Braket SDK) to compile and run circuits locally.

In [4]:
# No QCanvas; simulation is done by ValidatorAgent via QuantumSimulator (Braket SDK)
print("Simulation will be run by ValidatorAgent using the local Braket SDK.")

⚠️ Skipping simulation test - conversion was not successful.


## 1.4 Part 1 complete

Ready to use the Validator Agent in **local** mode (Amazon Braket SDK).

In [5]:
# Part 1 done. ValidatorAgent will compile + simulate code locally (no QCanvas).
print("=" * 50)
print("Part 1 complete. Ready for Validator Agent (local Braket SDK).")
print("=" * 50)

📤 Testing full pipeline (convert + execute)...



2026-03-15 02:02:39 | ERROR    | src.tools.qcanvas_client:convert_to_qasm:69 | Conversion error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/converter/convert (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


Success: False
Stage: conversion

❌ Pipeline failed at stage: conversion
   Error: Conversion error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/converter/convert (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

🎉 BACKEND TEST COMPLETE - Ready for Validation!


---

# Part 2: Validator Agent Testing

The Validator Agent runs in **local** mode: it compiles and executes code using the Amazon Braket SDK (BraketCompiler + QuantumSimulator).

## 2.1 Initialize Validator Agent

In [6]:
# First, add imports and setup RAG components
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever

# Initialize Knowledge Base with all datasets
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
kb.load_from_directory()
kb.index_entries()
print(f"Loaded {len(kb.entries)} KB entries for semantic validation")

# Create retriever
retriever = Retriever(knowledge_base=kb)

# Initialize in LOCAL mode: uses Amazon Braket SDK (no QCanvas)
validator = ValidatorAgent(mode="local", retriever=retriever)

print("✅ ValidatorAgent initialized (local Braket SDK, RAG support)")
...
print(f"   Mode: {validator.mode}")
print(f"   LLM Enabled: {validator.llm_enabled}")
print(f"   Ollama Model: {validator.ollama_model}")
print(f"   Default Shots: {validator.default_shots}")


2026-03-15 02:02:39,553 - botocore.credentials - INFO - credentials.py:1252 - Found credentials in environment variables.


2026-03-15 02:02:39 | INFO     | src.rag.embeddings:_init_aws:120 | Using AWS Nova Multimodal Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-15 02:02:39 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2026-03-15 02:02:39 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2026-03-15 02:02:39 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 1024
2026-03-15 02:02:39 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2026-03-15 02:02:39 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Hackathon\Amazons\data\knowledge_base\curated_designer_examples.jsonl
2026-03-15 02:02:39 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Hackathon\Amazons\data\knowledge_base\curated_designer_examples.jsonl
2026-03-15 02:02:39 | INFO     | src.rag.kno

KeyboardInterrupt: 

## 2.2 Validate Valid Code (Bell State)

In [ ]:
valid_code = """
from braket.circuits import Circuit

# Create a Bell state circuit
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1),
    braket.measure(q0, q1, key='result')
)
print(circuit)
"""

task = {
    "code": valid_code,
    "description": "Create a Bell state with two entangled qubits",
    "shots": 1024
}

print("🔍 Validating Bell state code...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('success'):
    results = result.get('results', {})
    print(f"\n📊 Results:")
    print(f"   Counts: {results.get('counts', {})}")
    print(f"   Probabilities: {results.get('probs', {})}")
else:
    print(f"\n🔴 Error: {result.get('error')}")

2025-12-07 15:45:46 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'11': 513, '00': 511}


🔍 Validating Bell state code...

✅ Validation PASSED
Stage: simulation

📊 Results:
   Counts: {'11': 513, '00': 511}
   Probabilities: None


## 2.3 Validate Invalid Code (with LLM Fix)

In [ ]:
# Code with missing import and missing measurement
invalid_code = """
# Missing from braket.circuits import Circuit!
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1)
)
"""

task = {
    "code": invalid_code,
    "description": "Create a simple quantum circuit with H and CNOT gates",
    "force_llm_fix": True
}

print("🔍 Validating broken code (missing import)...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('error'):
    print(f"\n🔴 Error: {result.get('error')}")

# Show LLM analysis
llm_analysis = result.get('llm_analysis', {})
if llm_analysis:
    print("\n🤖 LLM Analysis:")
    if llm_analysis.get('fixed_code'):
        print("\n📝 Fixed Code:")
        print("```python")
        print(llm_analysis['fixed_code'])
        print("```")
    if llm_analysis.get('analysis'):
        print(f"\n💡 Explanation: {llm_analysis['analysis']}")

2025-12-07 15:45:47 | ERROR    | src.tools.simulator:simulate:158 | Simulation error: Circuit has no measurements to sample.
2025-12-07 15:45:47 | WARNING  | src.agents.validator:_execute_local:438 | Simulation failed: Circuit has no measurements to sample.
2025-12-07 15:45:47 | INFO     | src.agents.validator:_execute_local:496 | Running LLM analysis for code fixing...


🔍 Validating broken code (missing import)...

✅ Validation PASSED
Stage: simulation_failed

🔴 Error: Circuit has no measurements to sample.

🤖 LLM Analysis:

📝 Fixed Code:
```python
from braket.circuits import Circuit

# Define qubits
q0, q1 = braket.LineQubit.range(2)

# Create a circuit with H and CNOT gates
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1)
)

# Add a measurement to the circuit
circuit.append(braket.measure(q0, q1))

# Simulate the circuit
simulator = braket.Simulator()
result = simulator.run(circuit, repetitions=1000)

# Print the results
print(result.histogram(key='q(0),q(1)'))
```

💡 Explanation: **Description:**
The original code lacked a measurement operation, which is necessary for sampling and obtaining results from a quantum circuit. I added a `braket.measure(q0, q1)` to measure both qubits at the end of the circuit. This allows the simulator to produce meaningful output when run.


## 2.4 Re-validate Fixed Code

In [ ]:
fixed_code = result.get('fixed_code') or result.get('llm_analysis', {}).get('fixed_code')

if fixed_code:
    print("🔍 Re-validating LLM-fixed code...")
    
    task_fixed = {
        "code": fixed_code,
        "description": "Fixed version of the circuit",
        "shots": 1024
    }
    
    result_fixed = validator.execute(task_fixed)
    
    print(f"\n{'✅' if result_fixed.get('success') else '❌'} Fixed code {'PASSED' if result_fixed.get('success') else 'FAILED'}")
    
    if result_fixed.get('success'):
        results = result_fixed.get('results', {})
        print(f"   Counts: {results.get('counts', {})}")
    else:
        print(f"   Error: {result_fixed.get('error')}")
else:
    print("⚠️ No fixed code available from previous step.")

2025-12-07 15:46:01 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'11': 534, '00': 490}


🔍 Re-validating LLM-fixed code...

✅ Fixed code PASSED
   Counts: {'11': 534, '00': 490}


---
# Part 3: Comprehensive Test Suite

Run the full suite of 20+ test cases to verify the Validator's robustness.

In [ ]:
from tests.test_validator_circuits import run_tests

# Run all test cases
df_results = run_tests(validator)

# Display summary
print("\n" + "="*40)
print("TEST SUITE SUMMARY")
print("="*40)
print(f"Total Tests: {len(df_results)}")
print(f"Passed: {len(df_results[df_results['Status'].str.contains('PASS')])}")
print(f"Failed: {len(df_results[df_results['Status'].str.contains('FAIL|ERROR')])}")

# Show full table
df_results

📝 Logging execution details to: D:\University\Uni\Semester 7\Generative AI\Project\Braket-RAG-Code-Assistant\tests\test_execution.log
🚀 Starting Validator Test Suite (22 cases)...

[1/22] Testing: Teleportation - Valid... 

15:46:01 | Simulation completed. Histogram: {'1': 550, '0': 474}
15:46:01 | Running LLM analysis for code fixing...


-> PASS (8.25s)
[2/22] Testing: Teleportation - Missing Import... -> PASS (Detected) (0.00s)
[3/22] Testing: Teleportation - Typo in Module... -> PASS (Detected) (0.00s)
[4/22] Testing: Teleportation - Undefined Variable... -> PASS (Detected) (0.00s)
[5/22] Testing: Teleportation - Syntax Error... -> PASS (Detected) (0.00s)
[6/22] Testing: Deutsch-Jozsa - Valid... 

15:46:09 | Simulation completed. Histogram: {'1': 1024}
15:46:09 | Running LLM analysis for code fixing...


-> PASS (8.63s)
[7/22] Testing: Deutsch-Jozsa - Indentation Error... -> PASS (Detected) (0.00s)
[8/22] Testing: Deutsch-Jozsa - Wrong Gate Usage... -> PASS (Detected) (0.00s)
[9/22] Testing: Deutsch-Jozsa - Logic/Type Error... -> PASS (Detected) (0.00s)
[10/22] Testing: Deutsch-Jozsa - Missing Measurement... 

...

15:46:30 | Simulation completed. Histogram: {'0': 1024}
15:46:30 | Running LLM analysis for code fixing...


-> PASS (6.93s)
[16/22] Testing: Grover - Valid... 

15:46:37 | Simulation completed. Histogram: {'1': 1024}
15:46:37 | Running LLM analysis for code fixing...


-> PASS (8.91s)
[17/22] Testing: Grover - Argument Mismatch... -> PASS (Detected) (0.00s)
[18/22] Testing: Grover - Variable Name Typo... -> PASS (Detected) (0.00s)
[19/22] Testing: Grover - Unsupported Gate... -> PASS (Detected) (0.00s)
[20/22] Testing: Grover - Invalid Append Type... -> PASS (Detected) (0.00s)
[21/22] Testing: Misc - Empty Code... -> PASS (Detected) (0.00s)
[22/22] Testing: Misc - Natural Language... -> PASS (Detected) (0.00s)

TEST SUITE SUMMARY
Total Tests: 22
Passed: 20
Failed: 2


,ID,Name,Description,Expected,Actual_Success,Has_Fix,Status,Error_Msg
0,1,Teleportation - Valid,Standard Teleportation circuit (Separate appen...,PASS,True,True,PASS,
1,2,Teleportation - Missing Import,Using invalid module to force import error,FAIL,False,False,PASS (Detected),No circuit found in code...
2,3,Teleportation - Typo in Module,Typo 'crique' instead of 'braket',FAIL,False,False,PASS (Detected),
3,4,Teleportation - Undefined Variable,Using q3 which is not defined,FAIL,False,False,PASS (Detected),
4,5,Teleportation - Syntax Error,Missing closing parenthesis,FAIL,False,False,PASS (Detected),
5,6,Deutsch-Jozsa - Valid,Standard DJ circuit for 2 qubits (Simplified),PASS,True,True,PASS,
6,7,Deutsch-Jozsa - Indentation Error,Python indentation error in loop,FAIL,False,False,PASS (Detected),
7,8,Deutsch-Jozsa - Wrong Gate Usage,Using parameters for non-parametric gate,FAIL,False,False,PASS (Detected),
8,9,Deutsch-Jozsa - Logic/Type Error,Trying to operate on integer instead of qubit,FAIL,False,False,PASS (Detected),
9,10,Deutsch-Jozsa - Missing Measurement,Circuit has no measurements,FAIL,True,True,FAIL (False Positive),Circuit has no measurements to sample....
